<div align="center">
<a href="https://colab.research.google.com/github/DubeyDivyansh/video_production_engine/blob/main/notebooks/01_avatar_knowledge_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# 🧠 Avatar Knowledge Base & Asset Calibration Studio
### Zero GPU Quota Consumed • Ephemeral-Proof • Google Drive Persistent Profile
</div>
---
Use this lightweight notebook to register your personal portrait photos, whole-body suit poses, and speech recordings.
The system automatically scores quality (Audio SNR, image sharpness, pose clarity) and saves your active profile to Google Drive.


## 1. Mount Google Drive
Persists avatar profiles across ephemeral Colab sessions under `/content/drive/MyDrive/video_gen_platform/`.


In [ ]:
from google.colab import drive
import os
try:
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/video_gen_platform'
    for sub in ['checkpoints', 'avatars', 'inputs', 'outputs']:
        os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)
    print(f'✅ Persistent Drive storage active at: {BASE_DIR}')
except Exception as e:
    print('⚠️ Drive mount notice (running in local / ephemeral mode):', e)


## 2. Clone Repository & Install Lightweight Scoring Tools


In [ ]:
!git clone https://github.com/DubeyDivyansh/video_production_engine.git /content/app 2>/dev/null || (rm -rf /content/temp_repo && git clone https://github.com/DubeyDivyansh/DubeyDivyansh.git /content/temp_repo && cp -r /content/temp_repo/video_production_engine /content/app)
%cd /content/app
!pip install -q soundfile pillow numpy gdown
import sys
if '/content/app' not in sys.path:
    sys.path.append('/content/app')
print('✅ Environment ready!')


## 3. Register Assets & Calibrate Avatar Profile (`divyansh`)
Registers:
1. Close-up face portrait
2. Whole-body suit posture image (with natural gestures/hands)
3. Google Drive lecture video (extracts optimal 7.5s speech reference prompt)


In [ ]:
from src.avatar.manager import AvatarManager
import json

manager = AvatarManager(use_drive=True)
avatar_id = 'divyansh'

# 1. Register close-up face portrait
print('[*] Registering close-up face portrait...')
!wget -q -O /content/my_face.jpeg https://files.catbox.moe/puyuek.jpeg
res_img = manager.register_media(avatar_id, '/content/my_face.jpeg', media_type='image')
print(f"    - Face Sharpness: {res_img['quality_score']:.2f}")

# 2. Register whole-body suit pose (with hands & body posture)
print('\n[*] Registering whole-body suit pose...')
suit_pose = '/content/app/assets/body_portraits/divyansh_suit_body_pose.jpg'
if os.path.exists(suit_pose):
    res_body = manager.register_media(avatar_id, suit_pose, media_type='body_pose')
    print(f"    - Body Pose Score: {res_body['quality_score']:.2f}")

# 3. Ingest lecture video from Google Drive & extract speech
DRIVE_LECTURE_URL = 'https://drive.google.com/file/d/1yCoOYz-Q-pzAaDYEWJUV1RihuiYJGQX2/view?usp=sharing'
print('\n[*] Ingesting speech audio from Google Drive lecture video...')
res_drive = manager.ingest_drive_url(
    avatar_id=avatar_id,
    drive_url_or_id=DRIVE_LECTURE_URL,
    media_type='video',
    extract_video_audio=True
)

# 4. Display Upgraded Avatar Profile
profile = manager.get_avatar(avatar_id)
print(f"\n🎉 Avatar Profile Calibrated: '{profile.avatar_id}'")
print(f"   Optimal Audio: {profile.optimal_audio_path} (Score: {profile.optimal_audio_score:.2f})")
print(f"   Optimal Face Image: {profile.optimal_image_path} (Score: {profile.optimal_image_score:.2f})")
print(f"   Optimal Body Pose: {profile.optimal_body_pose_path} (Score: {profile.optimal_body_pose_score:.2f})")
print(f"   Archived Assets: {len(profile.raw_assets)}")
